In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from utils import *

from prefilter_correction import *

In [2]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

In [3]:
prefilter_file = 'phi-fdt-pref_20250916T023002_V202607171248C_0569160250.txt'

In [4]:
folder = '/home/ulyanov/data/solo/phi/prefilter/calibration/2025-09-16/'
files = sorted(glob.glob(folder + '*.fits.gz'))

with fits.open(files[0]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

wv = read_wavelengths(header)

data = data - 0.4 * crop(dark, header)
data = correct_prefilter(data, header, prefilter_file)
binning = 8
data = rebin(data, binning)

In [5]:
def line(x, shift, sigma, depth, amplitude, gamma=0.053):
    from scipy.special import voigt_profile

    f = voigt_profile(x - shift, sigma, gamma)
    f /= voigt_profile(0, sigma, gamma)
    return (1 - depth * f) * amplitude


def fit_line(wavelength, intensity):
    from scipy.optimize import curve_fit
    params = curve_fit(line, wavelength, intensity,  p0=(wavelength[np.argmin(intensity)], 0.04, 0.3, np.max(intensity)))[0]
    fit = line(wavelength, *params)
    return fit, params

In [6]:
i,j = 40,90
temp = data[:,i,j]

fit, params = fit_line(wv, temp)
print(params)

plt.figure(figsize=(10,10))
plt.plot(wv, temp)
plt.plot(wv, fit)

plt.grid(True)
plt.tight_layout()

[6.17334301e+03 4.98430855e-02 2.84049879e-01 1.29111489e+04]


In [7]:
Shift = np.zeros_like(data[0])
Sigma = np.zeros_like(data[0])
Depth = np.zeros_like(data[0])
Amplitude = np.zeros_like(data[0])

for i in range(data.shape[-2]):
    for j in range(data.shape[-1]):
        try:
            fit, params = fit_line(wv, data[:,i,j])
            shift, sigma, depth, amplitude = params
        except:
            shift, sigma, depth, amplitude = np.nan, np.nan, np.nan, np.nan

        Shift[i,j] = shift
        Sigma[i,j] = sigma
        Depth[i,j] = depth
        Amplitude[i,j] = amplitude

/tmp/ipykernel_88302/2959957149.py:11: OptimizeWarning: Covariance of the parameters could not be estimated
  params = curve_fit(line, wavelength, intensity,  p0=(wavelength[np.argmin(intensity)], 0.04, 0.3, np.max(intensity)))[0]
/tmp/ipykernel_88302/2959957149.py:5: RuntimeWarning: invalid value encountered in divide
  f /= voigt_profile(0, sigma, gamma)


In [8]:
mask = data[0] > 1000

plt.figure(figsize=(12,10))
plt.imshow(np.nan_to_num(Sigma) * mask, 'inferno', vmin=0.035, vmax=0.07)
plt.colorbar(label=r'Line width, $\AA$')
plt.tight_layout()

In [9]:
mask = data[0] > 1000

plt.figure(figsize=(12,10))
plt.imshow(np.nan_to_num(Depth) * mask, 'inferno', vmin=0.25, vmax=0.35)
plt.colorbar(label='Line depth')
plt.tight_layout()